# 使用说明
- 本笔记本通过网络搜索收集 AI 产业与投资新闻（使用 ddgs），无需付费的 LLM API。
- 代码已写入邮箱与授权码，可直接运行；首次运行需配置 Python 通过代理访问网络（以获取新闻）。
- 如果想要更深度的分析（例如把多条新闻摘要后由 LLM 生成洞见），可添加可选的 LLM API 集成（OpenAI / Azure / 本地模型），但那会需要 API key/token。

新闻搜索遇到超时问题时，可在 PowerShell / bash 中为 Python 配置代理环境变量：
```powershell
# Windows PowerShell
$env:HTTP_PROXY = 'http://your-proxy:port'
$env:HTTPS_PROXY = 'http://your-proxy:port'
```
```bash
# Linux/macOS
export HTTP_PROXY='http://your-proxy:port'

export HTTPS_PROXY='http://your-proxy:port'```

In [ ]:
# =============== 配置（已写入明文凭证，注意安全） ===============
RECEIVE_EMAIL = '2846233882@qq.com'  # 已写入明文邮箱
EMAIL_AUTH_CODE = 'tpoqzsauvcvsdhdj'  # 已写入明文授权码
# ==========================================================

import time
import smtplib
from datetime import datetime
from email.mime.text import MIMEText
from ddgs import DDGS

# 1. 生成更聚焦的搜索提示词，旨在获取行业动态、投资趋势、模型能力和学习建议
def get_search_prompts():
    return [
        '请汇总最近一周内关于AI大模型、基础模型（foundation models）和生成式AI的核心进展、关键发布与评测结果。',
        '请列出近三个月内AI领域的主要融资事件、领投方、被投公司及投资金额范围，关注基础模型、AI芯片和行业化应用。',
        '评估当前主流大模型（如GPT系列、Gemini、Claude、文心一言等）在理解、生成、多模态能力上的相对优势与已知局限。',
        '指出近半年中值得关注的AI公司战略调整、并购或生态合作，及其对技术落地的影响。',
        '结合当前行业动态，给出接下来3-6个月内值得学习或实践的技能清单（工程与研究方向）。',
        '列举主要云厂商（AWS/GCP/Azure/阿里/腾讯/华为）在AI开发平台和模型部署方面的最新产品与差异。',
        '关注AI安全、治理与合规相关的政策与企业实践，列出近期影响较大的政策或白皮书。',
        '提供一份简短的行业速览：核心公司、关键技术、资金流向与短期风险。',
    ]

# 2. 通过关键词做快速新闻抓取（基于网络搜索，不依赖 LLM API）
def get_ai_news(days=7, max_per_kw=4):
    print("🔍 正在搜索全球AI最新动态和投资新闻...")
    # 改用英文关键词（中文关键词被 URL 编码后过长，容易超时）
    # 使用英文关键词可降低搜索失败率并加快速度
    keywords = [
        'AI model release 2026',           # 代替：AI 大模型 最新发布
        'AI funding investment 2026',      # 代替：AI 融资 投资 2026
        'OpenAI GPT latest',               # 代替：OpenAI 发布 GPT
        'Google Gemini news',              # 代替：Google Gemini 发布
        'Anthropic Claude update',         # 代替：Anthropic 产品 更新
        'AI chip investment news',         # 代替：AI 芯片 新 投资
        'AI regulation policy',            # 代替：生成式 AI 监管 政策
        'AI enterprise application'        # 代替：AI 企业化 应用 投资 合作
    ]
    all_news = []
    success_count = 0
    fail_count = 0
    
    for i, kw in enumerate(keywords, 1):
        try:
            print(f"  [{i}/8] 搜索 '{kw}' ... ", end='', flush=True)
            # 增加超时时间（默认 10 秒可能不够，改为 15 秒）
            ddgs = DDGS(timeout=15)
            results = list(ddgs.news(kw, max_results=max_per_kw))
            all_news.extend(results)
            print(f"✅ 获得 {len(results)} 条新闻")
            success_count += 1
        except Exception as e:
            print(f"⚠️ 错误：{type(e).__name__}")
            fail_count += 1
            time.sleep(1)  # 如果失败，稍作延迟后继续下一个
            continue
    
    print(f"\n  搜索完成：{success_count} 个关键词成功，{fail_count} 个失败")
    
    # 去重
    unique = []
    titles = set()
    for n in all_news:
        t = n.get('title','').strip()
        if t and t not in titles:
            titles.add(t)
            unique.append(n)
    
    return unique

# 3. 生成 HTML 报告
def build_html_report(news_list):
    today = datetime.now().strftime('%Y年%m月%d日')
    html = f'<html><head><meta charset="utf-8"><title>AI简报 {today}</title></head><body>'
    html += f'<h2>📌 AI产业与投资每日简报 {today}</h2><hr/>'
    if not news_list:
        html += '<p>未检索到新闻，请检查网络或关键词设置。</p>'
    else:
        html += f'<p><strong>共收集 {len(news_list)} 条新闻</strong></p><hr/>'
    for i, n in enumerate(news_list,1):
        title = n.get('title','无标题')
        url = n.get('url','')
        html += f'<p><b>{i}. {title}</b><br/><small>🔗 <a href="{url}">{url}</a></small></p><hr/>'
    html += '<p><small>本简报自动搜集整理，仅供学习参考。</small></p>'
    html += '</body></html>'
    return html

# 4. 发送邮件（使用环境变量或外部安全存储）
def send_html_email(content, smtp_server='smtp.qq.com', port=465):
    if not RECEIVE_EMAIL or not EMAIL_AUTH_CODE:
        print('❗ 未检测到邮件或授权码，请先设置环境变量 AI_REPORT_EMAIL / AI_REPORT_EMAIL_AUTH')
        return
    msg = MIMEText(content, 'html', 'utf-8')
    msg['Subject'] = f'📌 AI产业与投资每日简报 {datetime.now().strftime('%Y-%m-%d')}'
    msg['From'] = RECEIVE_EMAIL
    msg['To'] = RECEIVE_EMAIL
    try:
        server = smtplib.SMTP_SSL(smtp_server, port)
        server.login(RECEIVE_EMAIL, EMAIL_AUTH_CODE)
        server.send_message(msg)
        server.quit()
        print('✅ 精美可视化简报已发送至邮箱！')
    except Exception as e:
        print('❌ 发送失败：', e)

# 5. 运行示例（不包含任何硬编码凭证）
if __name__ == '__main__':
    print('=== 推荐搜索提示词 ===')
    for p in get_search_prompts():
        print('·', p)
    print()
    news = get_ai_news(days=7)
    report = build_html_report(news)
    send_html_email(report)
    print('🎉 本地运行完成！')


=== 推荐搜索提示词 ===
· 请汇总最近一周内关于AI大模型、基础模型（foundation models）和生成式AI的核心进展、关键发布与评测结果。
· 请列出近三个月内AI领域的主要融资事件、领投方、被投公司及投资金额范围，关注基础模型、AI芯片和行业化应用。
· 评估当前主流大模型（如GPT系列、Gemini、Claude、文心一言等）在理解、生成、多模态能力上的相对优势与已知局限。
· 指出近半年中值得关注的AI公司战略调整、并购或生态合作，及其对技术落地的影响。
· 结合当前行业动态，给出接下来3-6个月内值得学习或实践的技能清单（工程与研究方向）。
· 列举主要云厂商（AWS/GCP/Azure/阿里/腾讯/华为）在AI开发平台和模型部署方面的最新产品与差异。
· 关注AI安全、治理与合规相关的政策与企业实践，列出近期影响较大的政策或白皮书。
· 提供一份简短的行业速览：核心公司、关键技术、资金流向与短期风险。

🔍 正在搜索全球AI最新动态和投资新闻...
⚠️ 搜索关键词出错：AI 大模型 最新发布 -> ('error sending request for url (https://duckduckgo.com/?q=AI+%E5%A4%A7%E6%A8%A1%E5%9E%8B+%E6%9C%80%E6%96%B0%E5%8F%91%E5%B8%83)', 'https://duckduckgo.com/?q=AI+%E5%A4%A7%E6%A8%A1%E5%9E%8B+%E6%9C%80%E6%96%B0%E5%8F%91%E5%B8%83')
⚠️ 搜索关键词出错：AI 融资 投资 2026 -> ConnectError: ConnectError('error sending request for url (https://www.bing.com/news/infinitescrollajax?q=AI+%E8%9E%8D%E8%B5%84+%E6%8A%95%E8%B5%84+2026&InfiniteScroll=1&first=11&SFX=1&cc=us&setlang=en)', 'https://www.bing.com/news/infinitescrollajax?q=AI+%E8%9E%8D%E8%B5%84

KeyboardInterrupt: 